<img src="https://github.com/comet-ml/opik/blob/main/apps/opik-documentation/documentation/static/img/opik-logo.svg?raw=true" width="200" height="100" alt="Opik Logo">


# Opik Agentic Tracing, Datasets, & Experiments Workshop
### LangGraph Edition

## **Overview**
This session provides a brief introduction into **tracing** your agents with Opik as well as running **experiments** on standardized **datasets** to see how changes to your application affect it's performance. In this session you will:
- Configure Opik with your API Key and Workspace
- Build a CRM agent using **LangGraph** and **LangChain**
- Trace all LLM calls, tool nodes, and routing in Opik via the `OpikTracer` callback
- Create a dataset of user inputs and ground truth expected answers
- Run experiments on the dataset with 3 different prompts
- Compare the performance of the agent with each of the 3 prompts

## Setup & Configuration

In [ ]:
%pip install opik langchain-openai langgraph --q

Configure Opik with your API Key and personal workspace

In [ ]:
import opik
opik.configure(use_local=False, force=True)

In [ ]:
import os
import json
import getpass
from typing import TypedDict
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from opik.integrations.langchain import OpikTracer
from opik.evaluation import evaluate
from opik.evaluation.metrics import LevenshteinRatio, AnswerRelevance, Usefulness
from opik.evaluation.metrics import base_metric, score_result
import opik

PROJECT_NAME = "CRM-Chatbot-Agent-Opik"
os.environ["PROJECT_NAME"] = PROJECT_NAME

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

llm = ChatOpenAI(model="gpt-4o")


class CRMState(TypedDict):
    query: str
    system_prompt: str
    category: str
    context: str
    response: str

## Tracing our Agent

### Defining the Agent

This workshop uses a custom **CRM Agent** built with **LangGraph** and **LangChain**. The agent is designed to handle customer relationship queries by routing them to one of three specialized tool nodes:

1. **Contact Insight Tool** - Retrieves information about contacts, accounts, and deals
2. **Sales Analytics Tool** - Provides forecasts, KPIs, and performance metrics
3. **Fallback Tool** - Handles queries outside the CRM domain

The agent follows this architecture:

<img src="https://i.imgur.com/O8kDKmQ.png" width="400">

Each step of the agent is automatically traced via Opik's `OpikTracer` callback, giving you full visibility into the agent's decision-making process.

## Building the Agent with LangGraph

We will use **LangGraph** to define the agent as a state graph, with Opik's `OpikTracer` callback for automatic tracing. The agent consists of:
- A `CRMState` definition to track data through the pipeline
- A query classifier node to route requests
- Three specialized tool nodes (contact, sales, fallback)
- A RAG retrieval step
- A response formatter node

**Define RAG Context**

Static contact and sales context that will be retrieved based on the query classification.

In [ ]:
CONTACT_CONTEXT = {
  "customer": {
    "contact": {
      "name": "Jessica Lau",
      "title": "Head of IT Procurement",
      "email": "jessica.lau@greenleaftech.com",
      "status": "High-value prospect"
    },
    "company": "Greenleaf Tech",
    "industry": "Renewable Energy Tech",
    "deal_stage": "Proposal Sent",
    "deal_size_estimate": 85000,
    "lead_source": "Webinar sign-up",
    "lead_source_date": "2025-03-15",
    "last_contacted": "2025-05-12",
    "engagement": {
      "emails_opened": 6,
      "emails_sent": 7,
      "average_email_open_time_hours": 2,
      "outreach_responses": 3,
      "outreach_sent": 5,
      "demos_attended": ["2025-04-04", "2025-05-09"],
      "downloads": [{"title": "Energy Grid Optimization", "date": "2025-04-17"}]
    },
    "internal_notes": [
      "Interested in predictive maintenance features",
      "Has decision authority but awaiting CFO sign-off",
      "Expressed need to implement Q3 (starting July)"
    ]
  }
}

SALES_CONTEXT = {
  "customer": {
    "company": "NovaHealth Systems",
    "industry": "Healthcare Technology",
    "region": "Northeast US",
    "annual_revenue": 12000000,
    "account_manager": "Derek Chen",
    "deal_history": [
      {"deal_id": "DH-1007", "closed_date": "2024-11-20", "value": 45000, "product": "EHR Integration Suite", "stage": "Closed Won", "sales_cycle_days": 38},
      {"deal_id": "DH-1018", "closed_date": "2025-05-06", "value": 67000, "product": "AI Diagnostics Module", "stage": "Closed Won", "sales_cycle_days": 54}
    ],
    "current_pipeline": [
      {"deal_id": "DH-1029", "product": "Mobile Patient Portal", "stage": "Negotiation", "expected_close_date": "2025-06-28", "expected_value": 85000, "sales_cycle_days_so_far": 29}
    ],
    "metrics": {
      "total_closed_value_last_12_months": 112000,
      "average_sales_cycle_days": 46,
      "close_rate": 0.66,
      "average_deal_size": 56000
    },
    "last_activity": "2025-06-06"
  }
}

**Retrieve Documents (RAG)**

Returns the appropriate context based on the query category.

In [ ]:
def rag_lookup(category):
    if category == "contact":
        return json.dumps(CONTACT_CONTEXT, indent=4)
    elif category == "sales":
        return json.dumps(SALES_CONTEXT, indent=4)
    else:
        return "No context available."

**Define Tool Nodes**

Three specialized LangGraph nodes handle different query types:
- `contact_insight_tool` — information about people, accounts, and deals
- `sales_analytics_tool` — forecasts, KPIs, and performance metrics
- `fallback_tool` — handles out-of-scope queries

In [ ]:
def contact_insight_tool(state: CRMState) -> dict:
    context = rag_lookup("contact")
    return {"context": context}


def sales_analytics_tool(state: CRMState) -> dict:
    context = rag_lookup("sales")
    return {"context": context}


def fallback_tool(state: CRMState) -> dict:
    return {"context": "No relevant context available for this query."}

**Define Routing Logic**

Routes the classified query to the appropriate tool node. LangGraph will automatically generate the graph visualization in Opik.

In [ ]:
def route_query(state: CRMState) -> str:
    category = state["category"]
    if category == "contact":
        return "contact"
    elif category == "sales":
        return "sales"
    return "other"

**Classify Query Node**

Routes the user's question to the appropriate tool node based on the query type.

In [ ]:
def classify_query(state: CRMState) -> dict:
    query = state["query"]
    prompt = f"""
You are a routing assistant. Classify the following CRM query into one of:

- "contact" for information about people, accounts or deals
- "sales" for information about sales, pipelines, forecasts, revenue, KPIs or performance
- "other" for anything else

Respond with only one word: contact, sales or other.

Query: {query}
Category:"""
    response = llm.invoke([{"role": "user", "content": prompt}])
    return {"category": response.content.strip().lower()}

**Format Response Node**

Takes the retrieved context from the state and formats it into a user-friendly response.

In [ ]:
def format_response(state: CRMState) -> dict:
    prompt = f"""{state['system_prompt']}

        Context:
        {state['context']}

        Question: {state['query']}
        """
    response = llm.invoke([{"role": "user", "content": prompt}])
    return {"response": response.content.strip()}

**Build the Agent Graph**

Compile the **LangGraph StateGraph** that orchestrates the full agent flow:
1. Classifies the query
2. Routes to the appropriate tool node
3. Formats and returns the response

In [ ]:
graph = StateGraph(CRMState)

graph.add_node("classify_query", classify_query)
graph.add_node("contact_tool", contact_insight_tool)
graph.add_node("sales_tool", sales_analytics_tool)
graph.add_node("fallback_tool", fallback_tool)
graph.add_node("format_response", format_response)

graph.add_edge(START, "classify_query")
graph.add_conditional_edges("classify_query", route_query, {
    "contact": "contact_tool",
    "sales": "sales_tool",
    "other": "fallback_tool",
})
graph.add_edge("contact_tool", "format_response")
graph.add_edge("sales_tool", "format_response")
graph.add_edge("fallback_tool", "format_response")
graph.add_edge("format_response", END)

crm_agent = graph.compile()

## Test the Agent

Run a single query to verify the agent works and see the trace in Opik.

In [ ]:
system_prompt = opik.Prompt(
    name="CRM Agent - System Prompt",
    prompt="""
        You are a CRM agent. Please respond to the user query.
    """.rstrip().lstrip()
)

query = "Who is our point of contact at Green Leaf Tech?"

opik_tracer = OpikTracer(project_name=PROJECT_NAME)
res = crm_agent.invoke(
    {"query": query, "system_prompt": system_prompt.prompt},
    config={"callbacks": [opik_tracer]}
)

print(f"Category: {res['category']}")
print("-" * 40)
print(res['response'])

## **Datasets and Experiments**

## Creating a Dataset: CRM Agent Questions

Below we define a standard set of questions that we would like to evaluate the assistant on. The expected answers are concise — this will help us measure whether our prompt changes improve response quality.

In [ ]:
dataset_items = [
    {
        "question": "How many demos has Jessica Lau attended?",
        "expected_answer": "Jessica Lau attended 2 demos on 2025-04-04 and 2025-05-09."
    },
    {
        "question": "What is Jessica's current deal stage and estimated value?",
        "expected_answer": "Jessica's deal is in the Proposal Sent stage with an estimated value of $85,000."
    },
    {
        "question": "How engaged is Jessica based on email metrics?",
        "expected_answer": "Jessica opened 6 out of 7 emails sent, with an average open time of 2 hours."
    },
    {
        "question": "When was Jessica last contacted?",
        "expected_answer": "Jessica was last contacted on 2025-05-12."
    },
    {
        "question": "What do the internal notes say about Jessica's timeline?",
        "expected_answer": "Jessica has decision authority but is awaiting CFO sign-off. She wants to implement in Q3 starting July."
    },
    {
        "question": "What deals do we have with NovaHealth Systems?",
        "expected_answer": "NovaHealth Systems has three deals in our system: DH-1029 for the Mobile Patient Portal is currently in Negotiation at $85,000, DH-1007 for EHR Integration Suite is Closed Won at $45,000, and DH-1018 for AI Diagnostics Module is Closed Won at $67,000."
    },
    {
        "question": "What's the average deal size for NovaHealth?",
        "expected_answer": "The average deal size for NovaHealth Systems is approximately $65,667 based on their three deals totaling $197,000."
    },
    {
        "question": "What's the close rate for NovaHealth?",
        "expected_answer": "NovaHealth Systems has a close rate of 67%, with 2 out of 3 deals successfully closed."
    },
    {
        "question": "Show me the current sales pipeline.",
        "expected_answer": "The current active pipeline includes two deals: Greenleaf Tech in Proposal Sent stage valued at $85,000, and NovaHealth Systems in Negotiation stage valued at $85,000. Total active pipeline value is $170,000."
    },
    {
        "question": "What's our total closed revenue from NovaHealth?",
        "expected_answer": "Total closed revenue from NovaHealth Systems is $112,000, combining the EHR Integration Suite deal at $45,000 and the AI Diagnostics Module deal at $67,000."
    },
    {
        "question": "Who is our contact at NovaHealth?",
        "expected_answer": "We do not have any contacts on file at NovaHealth."
    },
]

In [ ]:
client = opik.Opik()

dataset = client.get_or_create_dataset(
    name="CRM_Agent_Questions",
    description="Questions about contacts and sales for evaluating response quality and length"
)

dataset.insert(dataset_items)

## Evaluating the Assistant

We will test the agent with 3 different system prompts to see if we can reduce response length while maintaining quality. We'll measure:
- **Levenshtein Ratio**: How close is the response to the expected answer (0-1)
- **Answer Relevance**: LLM-as-a-judge metric that scores how relevant the response is to the input question (0-1)
- **Usefulness**: LLM-as-a-judge metric that scores how useful the response is to the user (0-1)
- **Response Length**: Word count of the response

**Step 1: Fetch the dataset**

In [ ]:
dataset = client.get_dataset(name="CRM_Agent_Questions")

**Step 2: Define the system prompt to test**

In [ ]:
system_prompt = opik.Prompt(
    name="CRM Agent - System Prompt",
    prompt="""
        You are a helpful CRM assistant. Provide detailed, comprehensive answers
        to the user query using the provided context. Include all relevant information.
    """.rstrip().lstrip()
)

**Step 3: Define the evaluation task**

The evaluation task runs the full LangGraph agent pipeline for each question in the dataset.

In [ ]:
def make_evaluation_task(system_prompt):
    def evaluation_task(x):
        opik_tracer = OpikTracer(project_name=PROJECT_NAME)
        result = crm_agent.invoke(
            {"query": x["question"], "system_prompt": system_prompt.prompt},
            config={"callbacks": [opik_tracer]}
        )
        return {
            "response": result["response"],
            "context": result["context"]
        }
    return evaluation_task

**Step 4: Define metrics**

We use Opik's built-in `LevenshteinRatio` and `AnswerRelevance` metrics and a simple custom `ResponseLength` metric.

In [ ]:
class ResponseLength(base_metric.BaseMetric):
    def __init__(self, name: str = "response_length"):
        self.name = name

    def score(self, output: str, **kwargs):
        word_count = len(output.split())
        return score_result.ScoreResult(
            name="response_length",
            value=word_count,
            reason=f"{word_count} words"
        )

levenshtein_metric = LevenshteinRatio()
answer_relevance_metric = AnswerRelevance()
usefulness_metric = Usefulness()
length_metric = ResponseLength()

**Step 5: Run the evaluation**

In [ ]:
experiment_config = {
    "model": "gpt-4o",
    "prompt_version": "detailed"
}

res = evaluate(
    experiment_name="CRM Agent (LangGraph) - Detailed",
    dataset=dataset,
    experiment_config=experiment_config,
    project_name=PROJECT_NAME,
    task=make_evaluation_task(system_prompt),
    prompt=system_prompt,
    scoring_metrics=[levenshtein_metric, answer_relevance_metric, usefulness_metric, length_metric],
    scoring_key_mapping={
        "input": "question",
        "output": "response",
        "reference": "expected_answer",
    }
)

## Evaluating the Assistant (II)

Let's try a more concise prompt. The goal: reduce response length while maintaining similarity to expected answers.

In [ ]:
system_prompt = opik.Prompt(
    name="CRM Agent - System Prompt",
    prompt="""
        You are a CRM assistant. Answer in 1-2 sentences max. Be concise, but make sure you properly answer the query.
    """.rstrip().lstrip()
)

In [ ]:
experiment_config = {
    "model": "gpt-4o",
    "prompt_version": "concise"
}

res = evaluate(
    experiment_name="CRM Agent (LangGraph) - Concise",
    dataset=dataset,
    experiment_config=experiment_config,
    project_name=PROJECT_NAME,
    task=make_evaluation_task(system_prompt),
    prompt=system_prompt,
    scoring_metrics=[levenshtein_metric, answer_relevance_metric, usefulness_metric, length_metric],
    scoring_key_mapping={
        "input": "question",
        "output": "response",
        "reference": "expected_answer"
    }
)

## Evaluating the Assistant (III)

Let's try a third prompt that is very direct. The goal: reduce response length while maintaining similarity to expected answers.

In [ ]:
system_prompt = opik.Prompt(
    name="CRM Agent - System Prompt",
    prompt="""
        You are a CRM assistant. Answer in 1-2 sentences max. Be direct and concise. Only provide what is necessary, while still answering the query.
    """.rstrip().lstrip()
)

In [ ]:
experiment_config = {
    "model": "gpt-4o",
    "prompt_version": "direct"
}

res = evaluate(
    experiment_name="CRM Agent (LangGraph) - Direct",
    dataset=dataset,
    experiment_config=experiment_config,
    project_name=PROJECT_NAME,
    task=make_evaluation_task(system_prompt),
    prompt=system_prompt,
    scoring_metrics=[levenshtein_metric, answer_relevance_metric, usefulness_metric, length_metric],
    scoring_key_mapping={
        "input": "question",
        "output": "response",
        "reference": "expected_answer"
    }
)